In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
import time
import torch
import pandas as pd
import os

# import pygeohash as pgh

from collections import defaultdict
from torch_geometric.utils.convert import from_networkx
from torch_geometric.utils import to_undirected

from sklearn.model_selection import train_test_split

from tqdm.notebook import tqdm  # Use tqdm.notebook for Jupyter Notebook display

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import numpy as np
import torch
import random

def set_seed(seed=42):
    """Set the same seed for reproducibility across NumPy, PyTorch, and Python."""
    np.random.seed(seed)             # NumPy
    torch.manual_seed(seed)          # PyTorch CPU
    torch.cuda.manual_seed(seed)     # PyTorch GPU (if applicable)
    torch.cuda.manual_seed_all(seed) # If using multiple GPUs
    random.seed(seed)                # Python's built-in random module

    # Ensure deterministic behavior in PyTorch
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
SEED = 42

In [ ]:
set_seed(seed=SEED)

In [ ]:
base_path = '../data/processed'

In [ ]:
df = pd.read_csv(f'{base_path}/brightkite_all_events.csv', dtype={"uid": "Int64", "timestamp": "Int64", "other_uid": "Int64"})


In [ ]:
df

## Friend recommendation

In [ ]:
df_count = df.groupby(['uid', 'event_set']).agg(cnt=('event', 'count')).reset_index()
df_count

In [ ]:
df_freq = df_count.groupby(['event_set', 'cnt']).agg(freq=('uid', 'count')).reset_index()
df_freq

In [ ]:
dfr = df[df['event_set'] == 'relational'].reset_index()
dfr

In [ ]:
# 70-10-20 split, 0.125 * 0.8 = 0.1
dfr_train, dfr_test = train_test_split(dfr, test_size=0.2, random_state=SEED)
dfr_train, dfr_val = train_test_split(dfr_train, test_size=0.125, random_state=SEED)



In [ ]:
dfr_train = dfr_train.sort_values(['uid', 'other_uid'])[['uid', 'timestamp', 'event_set', 'event', 'other_uid']].reset_index(drop=True)
dfr_train

In [ ]:
dfr_test = dfr_test.sort_values(['uid', 'other_uid'])[['uid', 'timestamp', 'event_set', 'event', 'other_uid']].reset_index(drop=True)
dfr_test

In [ ]:
dfr_val = dfr_val.sort_values(['uid', 'other_uid'])[['uid', 'timestamp', 'event_set', 'event', 'other_uid']].reset_index(drop=True)
dfr_val

In [ ]:
dfp = df[df['event_set'] == 'personal'].reset_index(drop=True)
dfp 

In [ ]:
task_path = '../data/tasks/brightkite/friend_recommendation'
os.makedirs(task_path, exist_ok=True)

# dfp.to_csv(f'{task_path}/personal.csv', index=False)


In [ ]:
# dfr_train.to_csv(f'{task_path}/relational_train.csv', index=False)
# dfr_test.to_csv(f'{task_path}/relational_test.csv', index=False)
# dfr_val.to_csv(f'{task_path}/relational_val.csv', index=False)

In [ ]:
## check
# pd.read_csv(f'{task_path}/personal.csv')

### Negative sample

In [ ]:
NEG_SAMPLE_CNT = 50

In [ ]:
all_possible_ids = {int(uid) for uid in dfr['uid'].unique()}

def sample_negative(row, uid_to_positive, negative_count=10):
    """
    For a given row from df, sample negative ids.
    Negative samples are drawn from all_possible_ids excluding the positive ids
    for the given uid.
    """
    uid = row['uid']
    # Retrieve positive set for the uid, defaulting to empty set if not present.
    positive_set = uid_to_positive.get(uid, set())
    # Compute candidate negatives: all_possible_ids minus the positive set.
    candidates = list(all_possible_ids - positive_set)
    # Return negative_count samples, or fewer if not enough candidates.
    if len(candidates) < negative_count:
        return candidates
    else:
        return random.sample(candidates, negative_count)
    
    
def generate_negative_sample(dfr, uid_to_positive):
    negative_rows = []
    for idx, row in tqdm(dfr.iterrows(), total=len(dfr), desc="Generating negative samples"):
        neg_samples = sample_negative(row, uid_to_positive, negative_count=NEG_SAMPLE_CNT)
        for neg in neg_samples:
            negative_rows.append({
                'uid': row['uid'],
                'timestamp': row['timestamp'],
                'event_set': 'negative-' + row['event_set'],
                'event': row['event'],
                'other_uid': neg
            })

    # Create a new DataFrame from the negative sample rows.
    dfr_negative = pd.DataFrame(negative_rows)
    
    return dfr_negative
    


In [ ]:
# !pip install ipywidgets widgetsnbextension pandas-profiling
# !jupyter nbextension enable --py widgetsnbextension

In [ ]:
# validation
dfr_val_pos = pd.concat([dfr_val, dfr_train]).groupby('uid')['other_uid'].agg(set).reset_index()
uid_to_positive = dict(zip(dfr_val_pos['uid'], dfr_val_pos['other_uid']))
dfr_val_neg = generate_negative_sample(dfr_val, uid_to_positive)
dfr_val_neg


In [ ]:
# test
dfr_test_pos = pd.concat([dfr_test, dfr_train]).groupby('uid')['other_uid'].agg(set).reset_index()
uid_to_positive = dict(zip(dfr_test_pos['uid'], dfr_test_pos['other_uid']))
dfr_test_neg = generate_negative_sample(dfr_test, uid_to_positive)
dfr_test_neg

In [ ]:
# dfr_test_neg.to_csv(f'{task_path}/relational_test_negative_sample.csv', index=False)
# dfr_val_neg.to_csv(f'{task_path}/relational_val_negative_sample.csv', index=False)

## Checkin Recommendation

In [ ]:
dfp

In [ ]:
dfp['order'] = dfp.groupby("uid").cumcount()
dfp['max_order'] = dfp.groupby("uid")['order'].transform('max')
dfp['rev_order'] = dfp['max_order'] - dfp['order']

dfp['order_prop'] = dfp['order'] / dfp['max_order']
dfp['rev_order_prop'] = dfp['rev_order'] / dfp['max_order']

In [ ]:
# dfp['test_flag'] = ((dfp['rev_order_prop'] < 0.2) & (dfp['rev_order'] < 30))
# dfp['val_flag'] = ((dfp['test_flag'] != True) & (dfp['rev_order_prop'] < 0.3) & (dfp['rev_order'] < 45))

In [ ]:
dfp['test_flag'] = (((dfp['rev_order'] < np.ceil(0.2 * dfp['max_order'])) & (dfp['rev_order'] < 30)) | (dfp['rev_order'] == 0))
dfp['val_flag'] = ((dfp['test_flag'] != True) & (dfp['rev_order'] < (np.ceil(0.1 * dfp['max_order']) + np.ceil(0.2 * dfp['max_order']))) & (dfp['rev_order'] < 45))

In [ ]:
dfp

In [ ]:
dfp.groupby('uid').agg(cnt=('event', 'count')).reset_index().groupby('cnt').agg(cnt2=('uid', 'count'))

In [ ]:
dfp_train = dfp[((dfp['test_flag'] == False) & (dfp['val_flag'] == False))][['uid', 'timestamp', 'event_set', 'event', 'other_uid']].reset_index(drop=True)
dfp_train


In [ ]:
dfp_val = dfp[(dfp['val_flag'] == True)][['uid', 'timestamp', 'event_set', 'event', 'other_uid']].reset_index(drop=True)
dfp_val

In [ ]:
dfp_test = dfp[(dfp['test_flag'] == True)][['uid', 'timestamp', 'event_set', 'event', 'other_uid']].reset_index(drop=True)
dfp_test

In [ ]:
task_path = '../data/tasks/brightkite/checkin_prediction'
os.makedirs(task_path, exist_ok=True)

# dfr.to_csv(f'{task_path}/relational.csv', index=False)


In [ ]:
# dfp_train.to_csv(f'{task_path}/personal_train.csv', index=False)
# dfp_test.to_csv(f'{task_path}/personal_test.csv', index=False)
# dfp_val.to_csv(f'{task_path}/personal_val.csv', index=False)

### Negative sample

In [ ]:
NEG_SAMPLE_CNT = 10

In [ ]:
all_possible_ids = set(dfp['event'])

def sample_negative(dfp_row, uid_to_positive, negative_count=10):
    """
    For a given row from df, sample negative ids.
    Negative samples are drawn from all_possible_ids excluding the positive ids
    for the given uid.
    """
    uid = dfp_row['uid']
    positive_set = uid_to_positive.get(uid, set())
    candidates = list(all_possible_ids - positive_set)
    if len(candidates) < negative_count:
        return candidates
    else:
        return random.sample(candidates, negative_count)
    
def generate_negative_sample(dfp, uid_to_positive):
    negative_rows = []
    for idx, row in tqdm(dfp.iterrows(), total=len(dfp), desc="Generating negative samples for checkin"):
        neg_samples = sample_negative(row, uid_to_positive, negative_count=NEG_SAMPLE_CNT)
        for neg in neg_samples:
            negative_rows.append({
                'uid': row['uid'],
                'timestamp': None,
                'event_set': 'negative-' + row['event_set'],
                'event': neg,
                'other_uid': None
            })
    dfp_negative = pd.DataFrame(negative_rows)
    return dfp_negative

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

# Assume dfp is your DataFrame, uid_to_positive is a dictionary mapping uid to positive event sets,
# and all_possible_ids is the set of all geohash events from dfp['event'].
# For example:
# all_possible_ids = set(dfp['event'])

def generate_negative_sample(dfp, uid_to_positive, negative_count=10, base=3):
    """
    Group rows by uid so that for each uid we compute the candidate negatives once.
    Then, for each row in the group, we compute the weighted sampling based on prefix matching,
    with weight = base^(prefix_match_length). Each event (geohash8 string) is processed on the fly.
    
    Note: We cache only the candidate negatives per uid (which are reused across that uid's rows),
          not the uid-event pair, since those are likely unique.
    
    Parameters:
      - dfp: DataFrame with columns 'uid', 'event', and 'event_set'
      - uid_to_positive: Dictionary mapping uid to a set of positive event IDs.
      - negative_count: Number of negative samples to draw per row.
      - base: The exponentiation base for weighting.
    Returns:
      A new DataFrame with the negative samples.
    """
    negative_rows = []
    
    # Group rows by uid
    for uid, group in tqdm(dfp.groupby('uid'), total=dfp['uid'].nunique(),
                           desc="Processing groups by uid"):
        # Compute candidate negatives once per uid.
        positive_set = uid_to_positive.get(uid, set())
        candidates = list(all_possible_ids - positive_set)
        
        # Build candidate array as an (n_candidates, 8) array.
        candidate_array = np.frombuffer(''.join(candidates).encode('utf-8'),
                                        dtype='S1').reshape(-1, 8)
        n_candidates = candidate_array.shape[0]
        
        # For all rows in the group, convert their event (reference) into an (m, 8) array.
        refs = np.array([list(event) for event in group['event']])
        m = refs.shape[0]
        
        # Compute a boolean array comparing each reference against all candidates:
        # Shape will be (m, n_candidates, 8)
        match = (refs[:, None, :] == candidate_array[None, :, :])
        
        # Compute prefix-match lengths: find first mismatch for each (row, candidate) pair.
        mismatch = (~match).argmax(axis=2)
        # If all 8 characters match, argmax returns 0; fix those rows:
        all_match = np.all(match, axis=2)
        mismatch[all_match] = 8
        
        # For each row (event), compute weights for all candidates:
        # Weight = base^(prefix_match_length)
        weights = np.power(base, mismatch)  # shape: (m, n_candidates)
        
        # Process each row in the group (each row's event is unique, so no caching at uid-event level)
        for i, row in enumerate(group.itertuples(index=False)):
            w = weights[i]
            probs = w / w.sum()
            sampled_indices = np.random.choice(n_candidates, size=negative_count, replace=True, p=probs)
            for idx in sampled_indices:
                negative_rows.append({
                    'uid': row.uid,
                    'timestamp': None,
                    'event_set': 'negative-' + row.event_set,
                    'event': candidates[idx],
                    'other_uid': None
                })
    
    return pd.DataFrame(negative_rows)


In [ ]:
import numpy as np
import pandas as pd
import random
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor
from functools import partial

def compute_candidate(uid, pos_set, all_possible_ids):
    """Compute candidate Series for a given uid."""
    candidates = list(all_possible_ids - pos_set)
    return uid, pd.Series(candidates)

def sample_negative_for_row(row, uid_candidate_cache, negative_count=10):
    """
    For the given row, sample negative geohash strings from the candidate Series.
    Buckets are defined as:
      - Bucket 0: candidates that share fewer than 1 character with the current geohash.
      - Buckets 1 to 7: candidates that share at least k characters (for k=1,...,7).
    Negatives are sampled roughly equally from these 8 buckets, with any shortage filled from the overall candidate pool.
    """
    uid = row['uid']
    current = row['event']  # e.g., '9q9gn653'
    cand_series = uid_candidate_cache[uid]
    candidates = cand_series.tolist()
    if not candidates:
        return []
    
    # Convert candidates to a 2D NumPy array of shape (n_candidates, 8)
    cand_arr = np.array(candidates, dtype='S8').view('S1').reshape(-1, 8)
    # Convert current geohash to a 1x8 NumPy array using its characters.
    curr_arr = np.array(list(current), dtype='S1').reshape(1, 8)
    
    # Compute elementwise equality.
    eq = (cand_arr == curr_arr)
    # For each candidate, find the first index where it does NOT match.
    p = np.argmax(~eq, axis=1)
    
    # Define buckets:
    # Bucket 0: candidates with common prefix length < 1 (i.e. p == 0)
    bucket0 = np.where(p < 1)[0]
    # Buckets 1-7: candidates with common prefix length >= k.
    buckets = {k: np.where(p >= k)[0] for k in range(1, 8)}
    
    total_buckets = 8
    base = negative_count // total_buckets
    rem = negative_count % total_buckets
    sampled = set()
    
    # Sample from bucket 0:
    desired = base + (1 if 0 < rem else 0)
    if len(bucket0) > 0:
        chosen = np.random.choice(bucket0, desired, replace=False) if len(bucket0) >= desired else bucket0
        for i in chosen:
            sampled.add(candidates[i].decode('ascii') if isinstance(candidates[i], bytes) else candidates[i])
    
    # Sample from buckets 1 to 7:
    for k in range(1, 8):
        desired = base + (1 if k < rem else 0)
        idxs = buckets.get(k, [])
        if len(idxs) > 0:
            chosen = np.random.choice(idxs, desired, replace=False) if len(idxs) >= desired else idxs
            for i in chosen:
                sampled.add(candidates[i].decode('ascii') if isinstance(candidates[i], bytes) else candidates[i])
    
    sampled = list(sampled)
    # Fill in extra negatives from overall candidate pool if needed.
    if len(sampled) < negative_count:
        used = set(sampled)
        leftover = [s.decode('ascii') if isinstance(s, bytes) else s for s in candidates if (s.decode('ascii') if isinstance(s, bytes) else s) not in used]
        extra = negative_count - len(sampled)
        if len(leftover) <= extra:
            sampled.extend(leftover)
        else:
            sampled.extend(np.random.choice(leftover, extra, replace=False).tolist())
    
    return sampled

def sample_negative_for_row_wrapper(row, uid_candidate_cache, negative_count):
    """Wrapper that returns (uid, event_set, current, negatives) for parallel processing."""
    uid = row['uid']
    event_set = row['event_set']
    current = row['event']
    negatives = sample_negative_for_row(row, uid_candidate_cache, negative_count)
    return uid, event_set, current, negatives

def generate_negative_sample(df, uid_to_positive, negative_count=10, n_jobs=4):
    """
    Generate a DataFrame of negative samples.
    
    Parameters:
      - df: DataFrame with columns 'uid', 'event' (8-character geohash), and 'event_set'.
      - uid_to_positive: dict mapping uid to a set of positive geohash strings.
      - negative_count: number of negatives to sample per row.
      - n_jobs: number of parallel processes.
      
    Returns:
      A DataFrame with columns 'uid', 'timestamp', 'event_set', 'event', 'other_uid'
      (with timestamp and other_uid set to None).
    """
    # Precompute the global candidate pool.
    all_possible_ids = set(df['event'])
    
    # --- Parallel Candidate Pool Computation ---
    unique_uids = df['uid'].unique()
    uid_candidate_cache = {}
    with ProcessPoolExecutor(max_workers=n_jobs) as executor:
        futures = [executor.submit(compute_candidate, uid, uid_to_positive.get(uid, set()), all_possible_ids)
                   for uid in unique_uids]
        for future in tqdm(futures, desc="Computing candidate pools", total=len(futures)):
            uid, cand_series = future.result()
            uid_candidate_cache[uid] = cand_series
    
    # --- Parallel Negative Sampling ---
    neg_rows = []
    # Convert DataFrame rows to a list for parallel processing.
    rows = [row for _, row in df.iterrows()]
    process_func = partial(sample_negative_for_row_wrapper, uid_candidate_cache=uid_candidate_cache, negative_count=negative_count)
    
    with ProcessPoolExecutor(max_workers=n_jobs) as executor:
        # Use executor.map to preserve order.
        results = list(tqdm(executor.map(process_func, rows), total=len(rows), desc="Processing rows"))
    
    for uid, event_set, current, negatives in results:
        for neg in negatives:
            neg_rows.append({
                'uid': uid,
                'timestamp': None,
                'event_set': 'negative-' + event_set,
                'event': neg,
                'other_uid': None
            })
    
    return pd.DataFrame(neg_rows)

# Example usage:
# Suppose you have a DataFrame `df` with columns 'uid', 'event', and 'event_set'
# and a dictionary `uid_to_positive` mapping each uid to a set of positive geohash strings.
#
# Example:
# df = pd.DataFrame({
#     'uid': ['u1', 'u2', 'u1', 'u3'],
#     'event': ['9q9gn653', '9q9gn654', '9q9gn655', '9q9gn656'],
#     'event_set': ['A', 'B', 'A', 'C']
# })
#
# uid_to_positive = {
#     'u1': {'9q9gn653'},
#     'u2': {'9q9gn654'},
#     'u3': {'9q9gn656'}
# }
#
# df_negative = generate_negative_sample(df, uid_to_positive, negative_count=10, n_jobs=4)
# print(df_negative)


In [ ]:
dfp_val_pos = pd.concat([dfp_val, dfp_train]).groupby('uid')['event'].agg(set).reset_index()
uid_to_positive = dict(zip(dfp_val_pos['uid'], dfp_val_pos['event']))

In [ ]:
dfp_val.iloc[0:4]

In [ ]:
dfp_val.iloc[0:1]

In [ ]:
df2 = generate_negative_sample(dfp_val.iloc[0:4], uid_to_positive, 100, 4)

In [ ]:
len(df2)

In [ ]:
dfp_val_neg = generate_negative_sample(dfp_val, uid_to_positive, 100, 8)
dfp_val_neg

In [ ]:
df2

In [ ]:
dfp_val['event'][0]

In [ ]:
print(sum([ns.startswith('') for ns in df2['event'][:100]]))
print(sum([ns.startswith('9x') for ns in df2['event'][:100]]))
print(sum([ns.startswith('9xj4') for ns in df2['event'][:100]]))
print(sum([ns.startswith('9xj4p1') for ns in df2['event'][:100]]))

In [ ]:
df2

In [ ]:
dfpg = dfp.groupby('uid')

In [ ]:
dfpg.get_group(0)

In [ ]:
uid_candidate_cache = {}
for uid in tqdm(dfp_val.iloc[0:1]['uid'].unique(), desc="Precomputing candidate pools per uid"):
    pos_set = uid_to_positive.get(uid, set())
    # Candidates are all geohash strings minus this uid's positive ones.
    candidates = list(all_possible_ids - pos_set)
    uid_candidate_cache[uid] = pd.Series(candidates)

In [ ]:
negative_count = 100
row = dfp_val.iloc[0,:]

uid = row['uid']
current = row['event']  # e.g. '9q9gn653'
cand_series = uid_candidate_cache[uid]
candidates = cand_series.tolist()

# Convert candidates to a 2D NumPy array of shape (n_candidates, 8).
# (They are fixed-length 8 strings; note that np.array(..., dtype='S8') produces byte strings.)
cand_array = np.array(candidates, dtype='S8').view('S1').reshape(-1, 8)
# Convert current geohash into a 1x8 NumPy array (of byte strings).
curr_arr = np.array(list(current), dtype='S1').reshape(1, 8)

# Compare candidates with the current geohash (vectorized).
eq = (cand_array == curr_arr)  # shape (n_candidates, 8), boolean
# For each candidate, compute the common prefix length.
# (Since no candidate equals current, np.argmax(~eq, axis=1) works:
# for a candidate with no match in the first position, this returns 0;
# for one that matches in the first position, it returns the first position where it fails.)
p = np.argmax(~eq, axis=1)
# For candidates that match more than 0 positions, we have p >= 1, etc.
# (We assume no candidate fully matches.)

# Define buckets:
# Bucket 0: indices where common prefix length < 1 (i.e. p == 0).
bucket0 = np.where(p < 1)[0]
# For k from 1 to 7, bucket k = indices where common prefix length >= k.
buckets = {k: np.where(p >= k)[0] for k in range(1, 8)}

# We want to sample roughly equal negatives from each of 8 buckets.
# Set bucket 0 as defined above, and for k=1..7 use the corresponding buckets.
total_buckets = 8
base = negative_count // total_buckets
rem = negative_count % total_buckets
sampled = set()

# For bucket 0:
desired = base + (1 if 0 < rem else 0)
if len(bucket0) > 0:
    if len(bucket0) >= desired:
        chosen = np.random.choice(bucket0, desired, replace=False)
    else:
        chosen = bucket0
    for idx in chosen:
        # Decode byte string if needed.
        cand = candidates[idx]
        if isinstance(cand, bytes):
            cand = cand.decode('ascii')
        sampled.add(cand)

# For buckets 1 to 7:
for k in range(1, 8):
    desired = base + (1 if k < rem else 0)
    idxs = buckets.get(k, [])
    if len(idxs) > 0:
        if len(idxs) >= desired:
            chosen = np.random.choice(idxs, desired, replace=False)
        else:
            chosen = idxs
        for idx in chosen:
            cand = candidates[idx]
            if isinstance(cand, bytes):
                cand = cand.decode('ascii')
            sampled.add(cand)

sampled = list(sampled)
# If we still have too few negatives, fill the gap from the overall candidate pool.
if len(sampled) < negative_count:
    used = set(sampled)
    leftover = []
    for s in candidates:
        # Decode if necessary.
        s_decoded = s.decode('ascii') if isinstance(s, bytes) else s
        if s_decoded not in used:
            leftover.append(s_decoded)
    extra = negative_count - len(sampled)
    if len(leftover) <= extra:
        sampled.extend(leftover)
    else:
        sampled.extend(np.random.choice(leftover, extra, replace=False).tolist())


In [ ]:
current

In [ ]:
current

In [ ]:
current.encode('ascii')

In [ ]:
np.array(list(current), dtype='S1')

In [ ]:
curr_arr

In [ ]:
cand_array[1285]

In [ ]:
[(i,x) for i, x in enumerate(sampled) if x.startswith('9xj4')]

In [ ]:
df2

In [ ]:
# validation
dfp_val_pos = pd.concat([dfp_val, dfp_train]).groupby('uid')['event'].agg(set).reset_index()
uid_to_positive = dict(zip(dfp_val_pos['uid'], dfp_val_pos['event']))
dfp_val_neg = generate_negative_sample(dfp_val, uid_to_positive)
dfp_val_neg

In [ ]:
# test
dfp_test_pos = pd.concat([dfp_test, dfp_train]).groupby('uid')['event'].agg(set).reset_index()
uid_to_positive = dict(zip(dfp_test_pos['uid'], dfp_test_pos['event']))
dfp_test_neg = generate_negative_sample(dfp_test, uid_to_positive)
dfp_test_neg

In [ ]:
# dfp_test_neg.to_csv(f'{task_path}/personal_test_negative_sample.csv', index=False)
# dfp_val_neg.to_csv(f'{task_path}/personal_val_negative_sample.csv', index=False)